# Namespaces legb

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Namespaces und die LEGB-Regel</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 1: Fundament &nbsp;|&nbsp; Notebook 01b</p>
</div>
</div>

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Lernziele</span></div>

Nach diesem Notebook kannst du:

- erklären, was ein Namespace ist und wozu er dient
- die vier Scoping-Ebenen (Local, Enclosing, Global, Built-in) benennen und unterscheiden
- vorhersagen, welchen Wert Python einem Namen zur Laufzeit zuweist
- `global` und `nonlocal` korrekt einsetzen -- und begründen, wann das eine schlechte Idee ist
- typische Scoping-Fehler lesen, verstehen und beheben

**Voraussetzungen:** 01a -- Laufzeitmodell Python

**Legende**

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;"><span style="font-size:0.6rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.15em 0.5em;margin-right:0.65em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>Praxiswissen fuer den Kurs, nicht pruefungsrelevant</span></div>

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;"><span style="font-size:0.6rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.15em 0.5em;margin-right:0.65em;vertical-align:middle;white-space:nowrap;">PCAP</span>Inhalt wird in der Pruefung abgefragt</span></div>

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>Was ist ein Namespace?</span></div>

Ein **Namespace** ist eine Zuordnungstabelle: Name → Objekt.

Wenn du `x = 42` schreibst, legt Python den Eintrag `'x': <int-Objekt 42>` in einem Namespace ab. Wenn du später `print(x)` schreibst, schaut Python in einem Namespace nach, was `x` bedeutet.

Namespaces sind in Python schlicht Dictionaries. Das ist keine Metapher -- du kannst sie dir direkt anzeigen lassen:

In [ ]:
x = 100
name = 'Alice'

# globals() gibt den globalen Namespace als Dictionary zurueck
globaler_ns = globals()

print(type(globaler_ns))         # dict
print('x' in globaler_ns)        # True
print(globaler_ns['x'])          # 100
print(globaler_ns['name'])       # Alice

In [ ]:
def meine_funktion():
    lokal_a = 10
    lokal_b = 20
    # locals() gibt den lokalen Namespace zurueck
    print(locals())

meine_funktion()

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;"><span style="font-size:0.6rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.15em 0.5em;margin-right:0.65em;vertical-align:middle;white-space:nowrap;">PCAP 5.3</span>Lebensdauer von Namespaces</span></div>

Nicht alle Namespaces existieren gleich lang:

| Namespace | Entsteht | Verschwindet |
|-----------|----------|--------------|
| Built-in | beim Start des Interpreters | beim Beenden |
| Global (Modul) | beim Import / Start des Skripts | beim Beenden |
| Lokal (Funktion) | beim Funktionsaufruf | wenn die Funktion endet |
| Enclosing | bei äußerer Funktion | wenn äußere Funktion endet |

Das erklärt, warum lokale Variablen nach Funktionsende weg sind -- ihr Namespace wird einfach gelöscht.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>Scope: Wo sucht Python?</span></div>

Mehrere Namespaces können gleichzeitig existieren. Wenn Python einen Namen auflöst, muss es wissen, in welcher Reihenfolge es suchen soll. Diese Reihenfolge heißt **Scope** (Gültigkeitsbereich).

Python folgt dabei immer der **LEGB-Regel**:

```
L -- Local       (lokaler Namespace der aktuellen Funktion)
E -- Enclosing   (Namespaces umgebender Funktionen, von innen nach aussen)
G -- Global      (Namespace des Moduls / Skripts)
B -- Built-in    (Pythons eingebaute Namen: print, len, range, ...)
```

Python sucht immer von innen nach außen und nimmt den **ersten Treffer**.

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;"><span style="font-size:0.6rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.15em 0.5em;margin-right:0.65em;vertical-align:middle;white-space:nowrap;">PCAP 5.3</span>LEGB -- Uebersicht</span></div>

<div style="display:inline-flex;flex-direction:column;align-items:stretch;background:#080f17;border:1px solid #1a2e42;border-radius:8px;padding:1.5rem 2rem;margin:1rem 0;font-family:'Segoe UI',sans-serif;gap:0;">
  <div style="background:#0b1929;border:1px solid #1e4d6b;border-radius:5px;padding:0.5em 1.2em;color:#e8eaf6;font-size:0.9rem;font-weight:600;">
    <span style="color:#4fc3f7;">B</span> &mdash; Built-in: <span style="color:#7a7a90;font-weight:400;">print, len, range, int, ...</span>
    <div style="margin-top:0.6em;padding:0.5em 1em;background:#0a1622;border:1px solid #1e4d6b;border-radius:4px;">
      <span style="color:#4fc3f7;">G</span> &mdash; Global: <span style="color:#7a7a90;font-weight:400;">Variablen auf Modulebene</span>
      <div style="margin-top:0.6em;padding:0.5em 1em;background:#091420;border:1px solid #1e4d6b;border-radius:4px;">
        <span style="color:#4fc3f7;">E</span> &mdash; Enclosing: <span style="color:#7a7a90;font-weight:400;">aeussere Funktion</span>
        <div style="margin-top:0.6em;padding:0.5em 1em;background:#08121e;border:1px solid #4fc3f7;border-radius:4px;">
          <span style="color:#4fc3f7;">L</span> &mdash; Local: <span style="color:#7a7a90;font-weight:400;">diese Funktion</span>
        </div>
      </div>
    </div>
  </div>
</div>

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>Local Scope</span></div>

Variablen, die innerhalb einer Funktion definiert werden, leben im **lokalen Namespace** dieser Funktion. Sie sind von außen nicht sichtbar.

In [ ]:
def berechne():
    ergebnis = 42    # lokale Variable
    return ergebnis

print(berechne())    # 42

# ergebnis existiert ausserhalb der Funktion nicht
try:
    print(ergebnis)
except NameError as e:
    print(f'Fehler: {e}')

In [ ]:
# Eine lokale Variable kann einen globalen Namen ueberdecken (shadowing)
wert = 'global'

def zeige_wert():
    wert = 'lokal'   # neue lokale Variable -- die globale bleibt unveraendert
    print('Innerhalb:', wert)

zeige_wert()
print('Ausserhalb:', wert)

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>Global Scope</span></div>

Variablen auf Modulebene (d.h. außerhalb jeder Funktion) leben im **globalen Namespace**. Funktionen können globale Variablen lesen, aber nicht ohne weiteres überschreiben:

In [ ]:
zaehler = 0

def zeige_zaehler():
    print('Zaehler:', zaehler)   # Lesen ist problemlos

zeige_zaehler()

In [ ]:
zaehler = 0

def erhoehe():
    # Python sieht zaehler += 1 als Zuweisung und behandelt zaehler als lokal
    # Aber lokal existiert zaehler noch nicht -- UnboundLocalError
    try:
        zaehler += 1
    except UnboundLocalError as e:
        print(f'Fehler: {e}')

erhoehe()

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;"><span style="font-size:0.6rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.15em 0.5em;margin-right:0.65em;vertical-align:middle;white-space:nowrap;">PCAP 5.3</span>Das global-Schluesselwort</span></div>

Mit `global` erklaerst du, dass ein Name im globalen Namespace gesucht und veraendert werden soll:

In [ ]:
zaehler = 0

def erhoehe():
    global zaehler
    zaehler += 1

erhoehe()
erhoehe()
erhoehe()
print('Zaehler:', zaehler)   # 3

> [Kursinhalt] Wann global vermeiden
>
> `global` ist in Ordnung fuer Konfigurationswerte oder Konstanten. Fuer Zaehler, Zustaende oder akkumulierte Daten lieber eine Klasse verwenden.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>Enclosing Scope (Closures)</span></div>

Wenn eine Funktion innerhalb einer anderen Funktion definiert wird, hat die innere Funktion Zugriff auf die Variablen der aeusseren -- auch nachdem die aeussere Funktion beendet wurde. Das nennt man **Closure**.

In [ ]:
def erstelle_multiplier(faktor):
    # faktor lebt im Enclosing Scope von verdopple/verdreifache
    def multipliziere(zahl):
        return zahl * faktor   # faktor wird aus dem Enclosing Scope geholt
    return multipliziere

verdopple = erstelle_multiplier(2)
verdreifache = erstelle_multiplier(3)

print(verdopple(5))      # 10
print(verdreifache(5))   # 15

# erstelle_multiplier ist laengst beendet -- faktor lebt trotzdem noch
print(verdopple.__closure__[0].cell_contents)    # 2

In [ ]:
# Mehrere Enclosing-Ebenen: Python sucht von innen nach aussen
x = 'global'

def aussen():
    x = 'enclosing'
    def mitte():
        # kein eigenes x -- schaut in aussen
        def innen():
            print(x)   # findet 'enclosing' in der naechsten Enclosing-Ebene
        innen()
    mitte()

aussen()

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;"><span style="font-size:0.6rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.15em 0.5em;margin-right:0.65em;vertical-align:middle;white-space:nowrap;">PCAP 5.3</span>Das nonlocal-Schluesselwort</span></div>

Analog zu `global` gibt es `nonlocal` fuer den Enclosing Scope. Es erlaubt der inneren Funktion, eine Variable der aeusseren Funktion zu veraendern:

In [ ]:
def erstelle_zaehler():
    n = 0
    def erhoehe():
        nonlocal n
        n += 1
        return n
    return erhoehe

zaehler = erstelle_zaehler()
print(zaehler())   # 1
print(zaehler())   # 2
print(zaehler())   # 3

# Jeder Aufruf von erstelle_zaehler() erzeugt einen unabhaengigen Zaehler
anderer_zaehler = erstelle_zaehler()
print(anderer_zaehler())   # 1 -- faengt bei 0 an

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>Built-in Scope</span></div>

Der **Built-in Scope** ist immer vorhanden und enthaelt alle Namen, die Python von Haus aus mitbringt: `print`, `len`, `range`, `int`, `list`, `True`, `None` und viele mehr.

Er wird als letztes durchsucht -- das bedeutet: du kannst eingebaute Namen versehentlich ueberdecken.

In [ ]:
# Gefaehrliches Shadowing -- niemals so machen
# list = [1, 2, 3]   # ueberdeckt den eingebauten Typ list
# print(list([4, 5, 6]))   # TypeError: 'list' object is not callable

# Alle Built-ins koennen inspiziert werden
import builtins
alle_builtins = dir(builtins)
print(f'Anzahl Built-ins: {len(alle_builtins)}')
print('Einige davon:', alle_builtins[:15])

In [ ]:
# Demonstration: Built-in durch globale Variable verdeckt
print_original = print   # Referenz sichern

print = 'kein Print mehr'

try:
    print('Hallo')   # AttributeError oder TypeError
except TypeError as e:
    print_original(f'Fehler (erwartet): {e}')
finally:
    print = print_original   # Wiederherstellung

print('print ist wiederhergestellt')   # funktioniert wieder

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">PCAP 5.3</span>LEGB in der Praxis</span></div>

Die wichtigste Faehigkeit ist, den Wert eines Namens vorherzusagen, bevor du den Code ausfuehrst. Geh diese Beispiele durch, bevor du sie ausfuehrst:

In [ ]:
# Beispiel 1: Was wird ausgegeben?
x = 10

def f():
    x = 20
    def g():
        print(x)
    g()

f()
print(x)

# Deine Vorhersage vor dem Ausfuehren:
# f() gibt aus: ?
# print(x) gibt aus: ?

In [ ]:
# Beispiel 2: Was passiert hier?
wert = 5

def verdopple():
    wert = wert * 2   # Was passiert?
    return wert

try:
    print(verdopple())
except Exception as e:
    print(f'{type(e).__name__}: {e}')

# Erklaerung:
# Python sieht 'wert = ...' als Zuweisung und behandelt wert als lokal.
# Auf der rechten Seite wird wert gelesen, bevor es lokal zugewiesen wurde.

In [ ]:
# Beispiel 3: Klassischer Closure-Fallstrick
funktionen = []
for i in range(3):
    funktionen.append(lambda: i)

print([f() for f in funktionen])
# Erwartest du [0, 1, 2]? Was kommt wirklich heraus?

# Begruendung:
# lambda: i referenziert die Variable i, nicht ihren Wert zum Zeitpunkt der Definition.
# Nach der Schleife hat i den Wert 2 -- alle Lambdas sehen dasselbe i.

In [ ]:
# Loesung fuer den Closure-Fallstrick: Wert als Default-Argument binden
funktionen = []
for i in range(3):
    funktionen.append(lambda i=i: i)   # i=i: aktuellen Wert einfrieren

print([f() for f in funktionen])   # [0, 1, 2]

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;"><span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;font-family:'Segoe UI',sans-serif;white-space:nowrap;">Kursinhalt</span>Best Practices</span></div>

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Was man tun sollte</span></div>

- Variablen so lokal wie moeglich halten
- Werte als Parameter uebergeben statt globale Variablen lesen
- Rueckgabewerte verwenden statt `global`
- Namen fuer Konstanten auf Modulebene in GROSSBUCHSTABEN schreiben (`MAX_RETRIES = 3`) -- das ist Konvention, kein Schutz

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Was man vermeiden sollte</span></div>

- Built-in-Namen ueberdecken (`list`, `dict`, `print`, `id`, `type`, `input`, `sum`, `max`, `min`, ...)
- `global` fuer veraenderlichen Zustand
- Closures ueber Schleifenvariablen ohne Default-Argument-Trick

In [ ]:
# Schlecht: globaler veraenderlicher Zustand
punkte = 0

def punkt_geben_schlecht():
    global punkte
    punkte += 1

# Besser: Zustand als Parameter und Rueckgabewert
def punkt_geben_gut(punkte):
    return punkte + 1

aktuelle_punkte = 0
aktuelle_punkte = punkt_geben_gut(aktuelle_punkte)
aktuelle_punkte = punkt_geben_gut(aktuelle_punkte)
print('Punkte:', aktuelle_punkte)   # 2

# Oder als Klasse -- dazu spaeter mehr
class Punktestand:
    def __init__(self):
        self._punkte = 0
    def punkt_geben(self):
        self._punkte += 1
    @property
    def punkte(self):
        return self._punkte

stand = Punktestand()
stand.punkt_geben()
stand.punkt_geben()
print('Punkte:', stand.punkte)   # 2

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Zusammenfassung</span></div>

| Ebene | Wo | Schluesselwort zum Schreiben |
|-------|----|-----------------------------|
| Local | Aktuelle Funktion | (nichts noetig) |
| Enclosing | Aeussere Funktion | `nonlocal` |
| Global | Modul-/Skriptebene | `global` |
| Built-in | Pythons Standardnamen | (nicht veraendern) |

**Merksatz:** Python sucht einen Namen immer von innen nach aussen -- L vor E vor G vor B. Wer zuerst gefunden wird, gewinnt.

Lesen ist immer moeglich. Schreiben in aeussere Scopes erfordert `global` oder `nonlocal` -- und sollte gut begruendet sein.

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Aufgaben</span></div>

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Aufgabe 1 (Grundlagen)</span></div>

Bestimme fuer jeden Namen im folgenden Code, in welchem Scope er lebt (L / E / G / B). Schreibe deine Antwort als Kommentar.

In [ ]:
MAX = 100          # Scope: ?

def verarbeite(daten):   # 'verarbeite': ? , 'daten': ?
    ergebnis = []        # 'ergebnis': ?
    for wert in daten:   # 'wert': ?
        if wert < MAX:   # 'MAX': ? , 'wert' (hier): ?
            ergebnis.append(wert)   # 'append': ?
    return ergebnis

ausgabe = verarbeite([10, 200, 50, 150])   # 'ausgabe': ?
print(ausgabe)   # 'print': ?

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Aufgabe 2 (Vorhersage)</span></div>

Was gibt der folgende Code aus? Begruende deine Antwort, bevor du ihn ausfuehrst.

In [ ]:
a = 1

def aussen():
    a = 2
    def innen():
        a = 3
        print('innen:', a)
    innen()
    print('aussen:', a)

aussen()
print('global:', a)

# Deine Vorhersage (BEVOR du ausfuehrst):
# innen:  ?
# aussen: ?
# global: ?

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Aufgabe 3 (Fehler beheben)</span></div>

Der folgende Code enthaelt einen Scoping-Fehler. Finde ihn, erklaere warum er auftritt, und behebe ihn auf zwei verschiedene Arten.

In [ ]:
nachrichten = []

def nachricht_hinzufuegen(text):
    nachrichten = nachrichten + [text]   # Fehler!

nachricht_hinzufuegen('Hallo')
print(nachrichten)

# Loesung 1 (mit global -- nur wenn wirklich noetig):
# ...

# Loesung 2 (ohne global, eleganter):
# ...

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Aufgabe 4 (Closure bauen)</span></div>

Schreibe eine Funktion `erstelle_konto(startguthaben)`, die drei innere Funktionen zurueckgibt: `einzahlen(betrag)`, `auszahlen(betrag)` und `kontostand()`. Alle drei arbeiten auf demselben Kontostand. Verwende `nonlocal`.

In [ ]:
def erstelle_konto(startguthaben):
    # Dein Code hier
    pass


# Testcode
einzahlen, auszahlen, kontostand = erstelle_konto(100)

print(kontostand())     # 100
einzahlen(50)
print(kontostand())     # 150
auszahlen(30)
print(kontostand())     # 120

<div style="background:#0d1f2d;border-left:3px solid #1e4d6b;border-radius:3px;padding:0.55em 1em;margin:1.5rem 0 0.9rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(0.9rem,1.8vw,1.05rem);font-weight:600;color:#c5cae9;letter-spacing:-0.005em;">Aufgabe 5 (Vertiefung)</span></div>

Python hat eine eingebaute Funktion `vars()`. Was gibt sie zurueck?

Teste `vars()` an drei verschiedenen Stellen:
1. Auf Modulebene
2. Innerhalb einer Funktion
3. Mit einem Objekt als Argument (z.B. einem Modul wie `import math; vars(math)`)

Was ist der Unterschied zu `globals()` und `locals()`?

In [ ]:
# Deine Experimente hier
import math

# 1. Auf Modulebene

# 2. Innerhalb einer Funktion

# 3. Mit math als Argument

# Deine Schlussfolgerung:

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;"><span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">Weiter</span></div>

Im naechsten Notebook **01c -- Das Import-System** geht es darum, wie Python Module findet, laedt und cached -- und wie du das Import-Verhalten bei Bedarf kontrollieren kannst.